# Week 1 — Official User-Level Split (dayr4.2.csv)

This notebook generates the **official canonical train/validation/test split** for the Insider Threat Detection project, used by Week 1 (Random Forest baseline) and shared by Tier-1 / Tier-2 / Tier-3 so every experiment evaluates on the **exact same users**.

**Dataset:** `data/dayr4.2.csv` (the original engineered daily dataset) — **only** this dataset is used. The percentile dataset `dayr4.2-percentile30.pkl` is **not** used here (it is used only by the Tier-1 HTTP / File-Device / Email modality agents).

**Split reference:** the same user-level stratified split logic as `week3/tier1_score_generation.ipynb` (60% train / 20% validation / 20% test, stratified by the per-user insider label, `random_state=42`).

**Outputs (saved to `data/user_splits/`):**
- `train_users.npy`
- `val_users.npy`
- `test_users.npy`

> Note: rerunning this notebook regenerates the exact same split (deterministic, `random_state=42`), so it is safe and reproducible.


## 1. Setup

Resolve the project root by walking up to `pyproject.toml` (the project's portability convention).

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path
from sklearn.model_selection import train_test_split

_curr = Path.cwd()
while not (_curr / 'pyproject.toml').exists() and _curr != _curr.parent:
    _curr = _curr.parent
PROJECT_ROOT = _curr
DATA_DIR = PROJECT_ROOT / 'data'
SPLIT_DIR = DATA_DIR / 'user_splits'
SPLIT_DIR.mkdir(parents=True, exist_ok=True)

RAW_CSV = DATA_DIR / 'dayr4.2.csv'
print(f"Project root : {PROJECT_ROOT}")
print(f"Dataset path : {RAW_CSV.resolve()}")
print(f"Split output : {SPLIT_DIR.resolve()}")

Project root : C:\insider_threat_detection_project2
Dataset path : C:\insider_threat_detection_project2\data\dayr4.2.csv
Split output : C:\insider_threat_detection_project2\data\user_splits


## 2. Load dayr4.2.csv

Only the `user` and `insider` columns are needed to build the user-level split (the insider label per user determines stratification).

In [2]:
df = pd.read_csv(RAW_CSV, usecols=['user', 'insider'])
print(f"Loaded {RAW_CSV.name}: {len(df)} user-days, {df['user'].nunique()} users")

# Downcast to float32 if float (only if loaded as float)
for col in df.select_dtypes(include=['float64']).columns:
    df[col] = df[col].astype('float32')

Loaded dayr4.2.csv: 330452 user-days, 1000 users


## 3. Build the User-Level 60/20/20 Split

Exact logic from `week3/tier1_score_generation.ipynb`: stratify on the **max insider label per user**, then split users into 60% train / 40% temp, then halve temp into 20% validation / 20% test. `random_state=42` throughout, so the result is identical to the Tier-1 split.

In [3]:
# Stratified 60/20/20 user-level split (Train / Validation / Test)
user_labels = df.groupby('user')['insider'].max().astype(int)
unique_users = user_labels.index.values

# 1) 60% Train vs 40% Temp  (stratified by per-user insider label)
train_users, temp_users = train_test_split(
    unique_users, test_size=0.4, random_state=42, stratify=user_labels.values)

# 2) Temp split into two equal halves -> 20% Validation, 20% Test
val_users, test_users = train_test_split(
    temp_users, test_size=0.5, random_state=42,
    stratify=user_labels.loc[temp_users].values)

print(f"Train users: {len(train_users)} ({100*len(train_users)/len(unique_users):.0f}%)")
print(f"Val   users: {len(val_users)} ({100*len(val_users)/len(unique_users):.0f}%)")
print(f"Test  users: {len(test_users)} ({100*len(test_users)/len(unique_users):.0f}%)")

# Malicious users per split
ubin = (user_labels > 0).astype(int)
print(f"Malicious users -> train: {int(ubin.loc[train_users].sum())}, "
      f"val: {int(ubin.loc[val_users].sum())}, test: {int(ubin.loc[test_users].sum())}")

# Overlap check (must all be 0)
print(f"Overlap train/val={len(set(train_users)&set(val_users))}, "
      f"train/test={len(set(train_users)&set(test_users))}, "
      f"val/test={len(set(val_users)&set(test_users))}")

Train users: 600 (60%)
Val   users: 200 (20%)
Test  users: 200 (20%)
Malicious users -> train: 42, val: 14, test: 14
Overlap train/val=0, train/test=0, val/test=0


## 4. Save the Official Split

Save the user ID arrays as `.npy` files in `data/user_splits/` — the canonical project split.

In [4]:
np.save(SPLIT_DIR / 'train_users.npy', train_users)
np.save(SPLIT_DIR / 'val_users.npy',   val_users)
np.save(SPLIT_DIR / 'test_users.npy',  test_users)

print("Saved official split to:", SPLIT_DIR)
for f in ['train_users.npy', 'val_users.npy', 'test_users.npy']:
    p = SPLIT_DIR / f
    print(f"  {f:<18} {np.load(p).shape}  {p}")

Saved official split to: C:\insider_threat_detection_project2\data\user_splits
  train_users.npy    (600,)  C:\insider_threat_detection_project2\data\user_splits\train_users.npy
  val_users.npy      (200,)  C:\insider_threat_detection_project2\data\user_splits\val_users.npy
  test_users.npy     (200,)  C:\insider_threat_detection_project2\data\user_splits\test_users.npy


## 5. Verify (round-trip)

Reload the saved files and confirm the counts and zero user overlap — this proves the split is reproducible and correctly stored.

In [5]:
train_users2 = np.load(SPLIT_DIR / 'train_users.npy')
val_users2   = np.load(SPLIT_DIR / 'val_users.npy')
test_users2  = np.load(SPLIT_DIR / 'test_users.npy')

print("Reloaded counts:", len(train_users2), len(val_users2), len(test_users2))
print("Exact match (equal to in-memory):",
      np.array_equal(np.sort(train_users2), np.sort(train_users)),
      np.array_equal(np.sort(val_users2), np.sort(val_users)),
      np.array_equal(np.sort(test_users2), np.sort(test_users)))
print("User overlap (must all be 0):",
      len(set(train_users2) & set(val_users2)),
      len(set(train_users2) & set(test_users2)),
      len(set(val_users2) & set(test_users2)))
print("\n[OK] Official user-level split is saved and reproducible.")

Reloaded counts: 600 200 200
Exact match (equal to in-memory): True True True
User overlap (must all be 0): 0 0 0

[OK] Official user-level split is saved and reproducible.


## 6. Summary

- Generated the **official canonical user-level split** (60/20/20) from `data/dayr4.2.csv` only.
- Used the exact stratified split logic from `week3/tier1_score_generation.ipynb` (per-user insider label, `random_state=42`) — identical users to Tier-1 / Tier-2.
- No percentile dataset used; no modality agents involved.
- Saved `train_users.npy`, `val_users.npy`, `test_users.npy` to `data/user_splits/`.
- Downstream notebooks (Week 1 RF, Tier-1/2/3) load these files to ensure all experiments share the same users.